# Wie analysiere ich die dokumentierten Auffälligkeiten?

Wählt den Beispielordner oder euren Trainingsordner. Er enthält die Sollbefunde in `findings.csv` und die Dokumente in `anomaly_markdown/`. Mit zwei Funktionen könnt ihr anschließend einen Fehlertyp zählen und einzelne Textänderungen untersuchen.

In [1]:
import csv
from difflib import ndiff
from pathlib import Path

dataset_folder = Path('../examples')
with (dataset_folder / 'findings.csv').open(encoding='utf-8', newline='') as file:
    findings = list(csv.DictReader(file))

document_ids = []
for path in sorted((dataset_folder / 'anomaly_markdown').glob('*.md')):
    document_ids.append(path.stem.removeprefix('epd_').removesuffix('_with_anomalies'))

## Befunde je EPD zählen

`count_errors('spelling')` zählt die annotierten Schreibfehler pro Dokument und insgesamt. Probiert auch `'format'`, `'extra_word'` oder `'advertising'`. Alle Dokumente starten mit dem Wert 0, damit auch EPDs ohne Befund dieses Typs sichtbar bleiben.

In [2]:
def count_errors(error_type: str) -> None:
    if error_type not in ('spelling', 'format', 'extra_word', 'advertising'):
        raise ValueError(f'Unknown error type: {error_type}')
    counts = dict.fromkeys(document_ids, 0)
    for row in findings:
        if row['type'] == error_type:
            counts[row['id']] += 1

    for document_id, count in counts.items():
        print(f'id={document_id}  {error_type}={count}')
    print(f'Total {error_type}: {sum(counts.values())}')

In [3]:
count_errors('spelling')

id=101  spelling=1
id=102  spelling=0
Total spelling: 1


## Unterschiede pro Zeile ansehen

`show_changes('spelling')` zeigt die ersten drei passenden Befunde mit EPD-ID und Zeilennummer. Mit `limit=10` erhöht ihr die Zahl der Beispiele. Mit `show_context=True` erscheinen auch die vollständigen Original- und Befundzeilen.

`ndiff` vergleicht die beiden Wortfolgen. Wir geben nur Wörter mit `- ` (nur im Original) oder `+ ` (nur in der geprüften Fassung) aus. Satzzeichen bleiben am Wort. Ein Zusatzwort, eine Ersetzung und mehrere Änderungen in derselben Zeile lassen sich so mit derselben Funktion betrachten.

In [4]:
def show_changes(error_type: str, limit: int = 3, show_context: bool = False) -> None:
    matches = [row for row in findings if row['type'] == error_type]
    if not matches:
        print(f'No findings of type {error_type}.')

    for row in matches[:limit]:
        print('id={id}  line={line}  type={type}'.format_map(row))
        original = row['original_line']
        observed = row['anomaly_line']
        if show_context:
            print(f'Original: {original}')
            print(f'Observed: {observed}')

        if original == observed:
            print('Identical text')
        elif original.split() == observed.split():
            print(f'Whitespace change: {original!r} -> {observed!r}')
        else:
            for change in ndiff(original.split(), observed.split()):
                if change.startswith(('- ', '+ ')):
                    print(change)
        print()

In [5]:
show_changes('spelling')

id=101  line=3  type=spelling
- panel
+ pannel



Probiert auch eine annotierte Werbezeile. Werbung kann bereits im Original stehen. Dann erklärt der Text selbst den Befund.

In [6]:
show_changes('advertising', limit=1, show_context=True)

id=101  line=5  type=advertising
Original: Choose our leading panel today!
Observed: Choose our leading panel today!
Identical text



## Dieselben Befunde in Excel ansehen

1. Öffnet in Excel für Windows eine leere Arbeitsmappe. Wählt **Daten → Aus Text/CSV**. Je nach Menüband findet ihr den Eintrag unter **Daten → Daten abrufen → Aus Datei → Aus Text/CSV**.
2. Wählt die gewünschte `findings.csv`. Stellt in der Vorschau den Dateiursprung auf **65001: Unicode (UTF-8)** und das Trennzeichen auf **Komma**. Ihr solltet fünf Spalten sehen: `id`, `line`, `type`, `original_line`, `anomaly_line`.
3. Klickt auf **Laden**. Filtert in der Spalte `type` nach einem Fehlertyp und lest Original- und Befundzeile nebeneinander.

[Microsoft-Anleitung zum CSV-Import in Excel](https://support.microsoft.com/de-de/excel/get-started/import-or-export-text-txt-or-csv-files) · [Python-Dokumentation zu csv.DictReader](https://docs.python.org/3/library/csv.html#csv.DictReader) · [Wortfolgen mit difflib.ndiff vergleichen](https://docs.python.org/3/library/difflib.html#difflib.ndiff)